# Crawl comments theo danh sách channel ID (upload playlist)
Notebook này sẽ:
- Đọc danh sách channel ID từ file `channel_id.txt`
- Lấy upload playlist của từng channel
- Crawl danh sách video từ upload playlist
- Crawl comments của từng video
- Lưu 2 file CSV: videos và comments

In [14]:
import json
import os
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

print('Da import thu vien thanh cong.')

Da import thu vien thanh cong.


In [15]:
load_dotenv()

YOUTUBE_API_KEY = os.getenv('YOUTUBE_DATA_API_KEY')
if not YOUTUBE_API_KEY:
    raise ValueError('Khong tim thay YOUTUBE_DATA_API_KEY trong .env')

youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
print('Da khoi tao YouTube API client.')

Da khoi tao YouTube API client.


In [16]:
CHANNEL_IDS = [
    'UCrI4iNMPZ2vT_G-TqRO6yrw',
    'UCTF0ldaORTbCfx2ahvFfYWg',
    'UCiWyQp2HgKX2-WQUq11V8FA'
]

MAX_PLAYLIST_PAGES_PER_CHANNEL = 20
MAX_COMMENTS_PER_VIDEO = 200
INCLUDE_REPLIES = True
CHECKPOINT_FILE = 'comments_checkpoint.json'

channel_ids = list(dict.fromkeys([cid.strip() for cid in CHANNEL_IDS if cid and cid.strip()]))
if not channel_ids:
    raise ValueError('CHANNEL_IDS dang rong. Hay them it nhat 1 channel ID vao list CHANNEL_IDS.')

print(f'So channel IDs hop le: {len(channel_ids)}')
for i, cid in enumerate(channel_ids, 1):
    print(f'{i}. {cid}')

So channel IDs hop le: 3
1. UCrI4iNMPZ2vT_G-TqRO6yrw
2. UCTF0ldaORTbCfx2ahvFfYWg
3. UCiWyQp2HgKX2-WQUq11V8FA


In [17]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def make_uploads_playlist_id(channel_id: str) -> str:
    if not channel_id.startswith('UC') or len(channel_id) < 3:
        raise ValueError(f'Channel ID khong hop le de tao uploads playlist: {channel_id}')
    return 'UU' + channel_id[2:]


def fetch_videos_from_upload_playlist(youtube_client, channel_id: str, max_pages: int = 20):
    rows = []
    page_token = None
    page_count = 0
    uploads_playlist_id = make_uploads_playlist_id(channel_id)

    while page_count < max_pages:
        resp = youtube_client.playlistItems().list(
            part='snippet,contentDetails',
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=page_token
        ).execute()

        items = resp.get('items', [])
        if not items:
            break

        for it in items:
            snippet = it.get('snippet', {})
            content = it.get('contentDetails', {})
            video_id = content.get('videoId')
            if not video_id:
                continue

            rows.append({
                'channel_id': channel_id,
                'uploads_playlist_id': uploads_playlist_id,
                'video_id': video_id,
                'playlist_item_published_at': content.get('videoPublishedAt'),
                'playlist_position': snippet.get('position'),
                'playlist_item_title': snippet.get('title')
            })

        page_count += 1
        page_token = resp.get('nextPageToken')
        if not page_token:
            break

    return rows


def fetch_video_statistics(youtube_client, video_ids: list[str]):
    stats_rows = []
    for ids_batch in chunked(video_ids, 50):
        resp = youtube_client.videos().list(
            part='statistics',
            id=','.join(ids_batch),
            maxResults=50
        ).execute()

        for item in resp.get('items', []):
            stats = item.get('statistics', {})
            stats_rows.append({
                'video_id': item.get('id'),
                'view_count': stats.get('viewCount'),
                'like_count': stats.get('likeCount'),
                'comment_count': int(stats.get('commentCount', 0) or 0)
            })

    return pd.DataFrame(stats_rows)


def load_comment_checkpoint(file_path: str):
    if not os.path.exists(file_path):
        return None
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None


def save_comment_checkpoint(file_path: str, video_index: int, video_id: str, next_page_token: str | None):
    payload = {
        'video_index': video_index,
        'video_id': video_id,
        'next_page_token': next_page_token,
        'saved_at': datetime.now().isoformat()
    }
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def clear_comment_checkpoint(file_path: str):
    if os.path.exists(file_path):
        os.remove(file_path)


def get_video_comments(
    youtube_client,
    video_id: str,
    max_comments: int = 200,
    include_replies: bool = True,
    start_page_token: str | None = None,
    on_page_done=None
):
    comments = []
    page_token = start_page_token

    while len(comments) < max_comments:
        try:
            resp = youtube_client.commentThreads().list(
                part='snippet,replies',
                videoId=video_id,
                maxResults=100,
                pageToken=page_token,
                textFormat='plainText',
                order='time'
            ).execute()
        except Exception as e:
            msg = str(e)
            if 'commentsDisabled' in msg or 'videoNotFound' in msg:
                return comments
            print(f'  Loi comment video {video_id}: {msg[:120]}')
            return comments

        for item in resp.get('items', []):
            top = item['snippet']['topLevelComment']['snippet']
            top_id = item['snippet']['topLevelComment']['id']

            comments.append({
                'video_id': video_id,
                'comment_id': top_id,
                'parent_id': None,
                'is_reply': False,
                'author': top.get('authorDisplayName'),
                'text': top.get('textDisplay'),
                'like_count': top.get('likeCount'),
                'published_at': top.get('publishedAt'),
                'updated_at': top.get('updatedAt')
            })

            if include_replies and 'replies' in item:
                for rep in item['replies'].get('comments', []):
                    rs = rep.get('snippet', {})
                    comments.append({
                        'video_id': video_id,
                        'comment_id': rep.get('id'),
                        'parent_id': top_id,
                        'is_reply': True,
                        'author': rs.get('authorDisplayName'),
                        'text': rs.get('textDisplay'),
                        'like_count': rs.get('likeCount'),
                        'published_at': rs.get('publishedAt'),
                        'updated_at': rs.get('updatedAt')
                    })

            if len(comments) >= max_comments:
                break

        page_token = resp.get('nextPageToken')
        if on_page_done:
            on_page_done(page_token)

        if not page_token:
            break

    return comments

In [18]:
all_video_rows = []
channel_errors = []

print('Bat dau crawl videos tu upload playlist...')
for i, channel_id in enumerate(channel_ids, 1):
    print(f'[{i}/{len(channel_ids)}] Channel: {channel_id}')
    try:
        rows = fetch_videos_from_upload_playlist(
            youtube_client=youtube,
            channel_id=channel_id,
            max_pages=MAX_PLAYLIST_PAGES_PER_CHANNEL
        )
        all_video_rows.extend(rows)
        print(f'  Uploads playlist: {make_uploads_playlist_id(channel_id)}')
        print(f'  So video lay duoc: {len(rows)}')
    except Exception as e:
        channel_errors.append({'channel_id': channel_id, 'error': str(e)})
        print(f'  Loi: {str(e)[:120]}')

df_videos = pd.DataFrame(all_video_rows)
if not df_videos.empty:
    df_videos = df_videos.drop_duplicates(subset=['video_id']).reset_index(drop=True)

    stats_df = fetch_video_statistics(
        youtube_client=youtube,
        video_ids=df_videos['video_id'].dropna().astype(str).unique().tolist()
    )

    if not stats_df.empty:
        df_videos = df_videos.merge(stats_df, on='video_id', how='left')
        df_videos['comment_count'] = pd.to_numeric(df_videos['comment_count'], errors='coerce').fillna(0).astype(int)
        df_videos = df_videos[df_videos['comment_count'] > 0].reset_index(drop=True)

print('---')
print(f'Tong so video unique (comment_count > 0): {len(df_videos)}')
print(f'So channel loi: {len(channel_errors)}')
if channel_errors:
    print(pd.DataFrame(channel_errors).head())

Bat dau crawl videos tu upload playlist...
[1/3] Channel: UCrI4iNMPZ2vT_G-TqRO6yrw
  Uploads playlist: UUrI4iNMPZ2vT_G-TqRO6yrw
  So video lay duoc: 1000
[2/3] Channel: UCTF0ldaORTbCfx2ahvFfYWg
  Uploads playlist: UUTF0ldaORTbCfx2ahvFfYWg
  So video lay duoc: 1000
[3/3] Channel: UCiWyQp2HgKX2-WQUq11V8FA
  Uploads playlist: UUiWyQp2HgKX2-WQUq11V8FA
  So video lay duoc: 1000
---
Tong so video unique (comment_count > 0): 1922
So channel loi: 0


In [19]:
all_comments = []
video_ids = df_videos['video_id'].dropna().astype(str).unique().tolist() if not df_videos.empty else []

checkpoint = load_comment_checkpoint(CHECKPOINT_FILE)
start_index = 0
start_page_token = None

if checkpoint:
    cp_video_index = int(checkpoint.get('video_index', 0) or 0)
    cp_video_id = checkpoint.get('video_id')
    cp_next_page_token = checkpoint.get('next_page_token')

    if 0 <= cp_video_index < len(video_ids) and cp_video_id == video_ids[cp_video_index]:
        start_index = cp_video_index
        start_page_token = cp_next_page_token
        print(f"Resume tu checkpoint: index={start_index}, video_id={cp_video_id}, nextPageToken={cp_next_page_token}")
    else:
        print('Checkpoint khong con hop le voi danh sach video hien tai, se crawl lai tu dau.')

print(f'So video se crawl comments: {len(video_ids)}')

for idx in range(start_index, len(video_ids)):
    vid = video_ids[idx]
    token_for_this_video = start_page_token if idx == start_index else None

    if idx == start_index or (idx + 1) % 20 == 0:
        print(f'[{idx + 1}/{len(video_ids)}] Dang crawl comments...')

    def _on_page_done(next_token, current_idx=idx, current_vid=vid):
        save_comment_checkpoint(
            file_path=CHECKPOINT_FILE,
            video_index=current_idx,
            video_id=current_vid,
            next_page_token=next_token
        )

    comments = get_video_comments(
        youtube_client=youtube,
        video_id=vid,
        max_comments=MAX_COMMENTS_PER_VIDEO,
        include_replies=INCLUDE_REPLIES,
        start_page_token=token_for_this_video,
        on_page_done=_on_page_done
    )
    all_comments.extend(comments)

    next_idx = idx + 1
    if next_idx < len(video_ids):
        save_comment_checkpoint(
            file_path=CHECKPOINT_FILE,
            video_index=next_idx,
            video_id=video_ids[next_idx],
            next_page_token=None
        )

clear_comment_checkpoint(CHECKPOINT_FILE)

df_comments = pd.DataFrame(all_comments)
print('---')
print(f'Tong so comments: {len(df_comments)}')
if not df_comments.empty:
    print(df_comments.head(3))

So video se crawl comments: 1922
[1/1922] Dang crawl comments...
[20/1922] Dang crawl comments...
[40/1922] Dang crawl comments...
[60/1922] Dang crawl comments...
[80/1922] Dang crawl comments...
[100/1922] Dang crawl comments...
[120/1922] Dang crawl comments...
[140/1922] Dang crawl comments...
[160/1922] Dang crawl comments...
[180/1922] Dang crawl comments...
[200/1922] Dang crawl comments...
[220/1922] Dang crawl comments...
[240/1922] Dang crawl comments...
[260/1922] Dang crawl comments...
[280/1922] Dang crawl comments...
[300/1922] Dang crawl comments...
[320/1922] Dang crawl comments...
[340/1922] Dang crawl comments...
[360/1922] Dang crawl comments...
[380/1922] Dang crawl comments...
[400/1922] Dang crawl comments...
[420/1922] Dang crawl comments...
[440/1922] Dang crawl comments...
[460/1922] Dang crawl comments...
[480/1922] Dang crawl comments...
[500/1922] Dang crawl comments...
[520/1922] Dang crawl comments...
[540/1922] Dang crawl comments...
[560/1922] Dang crawl

In [20]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
videos_filename = f'videos_by_channel_uploads_{timestamp}.csv'
comments_filename = f'comments_by_channel_uploads_{timestamp}.csv'

df_videos.to_csv(videos_filename, index=False, encoding='utf-8-sig')
print(f'Da luu videos: {videos_filename} (rows={len(df_videos)})')

if not df_comments.empty:
    df_comments.to_csv(comments_filename, index=False, encoding='utf-8-sig')
    print(f'Da luu comments: {comments_filename} (rows={len(df_comments)})')
else:
    print('Khong co comments de luu.')

Da luu videos: videos_by_channel_uploads_20260324_111225.csv (rows=1922)
Da luu comments: comments_by_channel_uploads_20260324_111225.csv (rows=80520)


In [21]:
print('Preview videos:')
display(df_videos.head(5) if not df_videos.empty else 'Khong co du lieu video')

print('\nPreview comments:')
display(df_comments.head(5) if not df_comments.empty else 'Khong co du lieu comment')

Preview videos:


,channel_id,uploads_playlist_id,video_id,playlist_item_published_at,playlist_position,playlist_item_title,view_count,like_count,comment_count
0,UCrI4iNMPZ2vT_G-TqRO6yrw,UUrI4iNMPZ2vT_G-TqRO6yrw,_9LjdaGw0YQ,2026-03-23T03:00:37Z,1,"Điểm tin sáng 23/3: Man City vô địch Cúp Liên đoàn Anh, Tottenham nối dài chuỗi thất vọng",2491,43,1
1,UCrI4iNMPZ2vT_G-TqRO6yrw,UUrI4iNMPZ2vT_G-TqRO6yrw,fcekVpU7-ak,2026-03-23T01:14:45Z,2,Trực tiếp | SHB Đà Nẵng vs CLB Công An Hà Nội | Bình luận sau trận,2734,46,1
2,UCrI4iNMPZ2vT_G-TqRO6yrw,UUrI4iNMPZ2vT_G-TqRO6yrw,tOX9MBKCFnY,2026-03-22T03:41:51Z,4,Điểm tin 22/3 | Chelsea thảm bại trước Everton; Sự chênh lệch trên sân Hàng Đẫy,2767,41,1
3,UCrI4iNMPZ2vT_G-TqRO6yrw,UUrI4iNMPZ2vT_G-TqRO6yrw,l972HFieg8s,2026-03-21T23:44:31Z,5,TRỰC TIẾP bình luận sau trận: ĐT Australia 0-1 ĐT Nhật Bản | Chung kết AFC Women Cup 2026,5734,57,2
4,UCrI4iNMPZ2vT_G-TqRO6yrw,UUrI4iNMPZ2vT_G-TqRO6yrw,o04biJT8wIc,2026-03-21T09:03:45Z,7,TRỰC TIẾP bình luận trước trận: ĐT Australia vs ĐT Nhật Bản | Chung kết AFC Women Cup 2026,9927,40,1



Preview comments:


,video_id,comment_id,parent_id,is_reply,author,text,like_count,published_at,updated_at
0,_9LjdaGw0YQ,UgwHTdGZkkpymKq3qVJ4AaABAg,NaN,False,@ThànhNguyễn-e1g5w,Đầu nè,0,2026-03-23T04:33:47Z,2026-03-23T04:33:47Z
1,fcekVpU7-ak,UgzCViGuSmyNTkmfZTx4AaABAg,NaN,False,@EKAVODKHA,Selamat lebaran kawann🤝🤝🤝,0,2026-03-22T19:42:27Z,2026-03-22T19:42:27Z
2,tOX9MBKCFnY,UgzHJAKMABtCiaDPKXp4AaABAg,NaN,False,@ĐứcLê-m9g8s,Chương trình thể thao hay đó,1,2026-03-22T04:18:57Z,2026-03-22T04:18:57Z
3,l972HFieg8s,Ugz9WO7v-JCX7Osw-Vx4AaABAg,NaN,False,@sodium-natri-11,Úc chơi sân nhà vẫn thua😅😅,2,2026-03-21T13:50:17Z,2026-03-21T13:50:17Z
4,l972HFieg8s,UgxD9hsLJHEvo7hwgRZ4AaABAg,NaN,False,@zn-invention2021,"2026 này, bóng đá Nhật Bản đã có những giải đấu thành công như U23 Nhật Bản vô địch U23 Châu Á sau khi thắng dễ dàng...",6,2026-03-21T11:48:04Z,2026-03-21T11:48:04Z
